# Pronósticos basados en series de tiempo
## Ejercicio: Número de Ocupados — 13 Ciudades de Colombia
### Maestría en Ciencia de Datos — Universidad Javeriana Cali
**Estudiante:** Juan Sebastian Torres Romero

## **1. Carga de paquetes**

In [ ]:
import sys
!{sys.executable} -m pip install --quiet numpy pandas matplotlib scikit-learn statsmodels openpyxl

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error

## **2. Carga de datos**

In [ ]:
data = pd.read_excel(
    "https://raw.githubusercontent.com/dagudelo30/Series-de-tiempo---Javeriana-Cali/main/intro-moving_average/datosEmpleo.xlsx",
    index_col='mes', parse_dates=True
)
data.head()

In [ ]:
print(data.shape)

In [ ]:
plt.figure(figsize=(12, 5))
plt.title("Numero de ocupados (miles) por mes")
plt.xlabel("Mes")
plt.ylabel("Numero de ocupados (miles)")
plt.plot(data[["Ocupados"]])
plt.grid()
plt.show()

## **3. Protocolo de evaluacion**

Se reservan los ultimos 14 meses como conjunto de prueba fuera de muestra.

In [ ]:
train_len = 208
train_ocup = data[["Ocupados"]][:train_len]
test_ocup  = data[["Ocupados"]][train_len:]

print(f"Entrenamiento: {train_ocup.index[0].strftime('%Y-%m')} a {train_ocup.index[-1].strftime('%Y-%m')} (n={len(train_ocup)})")
print(f"Prueba:        {test_ocup.index[0].strftime('%Y-%m')} a {test_ocup.index[-1].strftime('%Y-%m')} (n={len(test_ocup)})")

fig = plt.figure(figsize=(12, 5))
plt.plot(train_ocup, label="Datos Entrenamiento")
plt.plot(test_ocup,  label="Datos de Prueba")
plt.legend()
plt.grid()
plt.show()

In [ ]:
train_ocup

In [ ]:
test_ocup

## **4. Promedio movil**

El promedio movil esta dado por:

$$ F_{t+1} = \frac{Y_t + Y_{t-1} + \cdots + Y_{t-(k-1)}}{k} $$

Se evaluan ordenes w = 2, 3, 4 y 5.

In [ ]:
# Sin considerar el dato actual
ma_2 = train_ocup.shift().rolling(2, min_periods=2).mean()
ma_3 = train_ocup.shift().rolling(3, min_periods=2).mean()
ma_4 = train_ocup.shift().rolling(4, min_periods=2).mean()
ma_5 = train_ocup.shift().rolling(5, min_periods=2).mean()

In [ ]:
# Funcion para pronosticar h pasos adelante con promedio movil de orden w
def fore_ma(datos, w, h):
    data = datos.copy()
    for x in range(1, h+1):
        ind   = data.index[-1]
        value = ind + pd.DateOffset(months=1)
        data.loc[value] = data[-w:].mean()
    return data[-h:]

In [ ]:
h_test = len(test_ocup)

ma_2_f = fore_ma(train_ocup, 2, h_test)
ma_3_f = fore_ma(train_ocup, 3, h_test)
ma_4_f = fore_ma(train_ocup, 4, h_test)
ma_5_f = fore_ma(train_ocup, 5, h_test)

In [ ]:
test_ocup

In [ ]:
ma_2_f

In [ ]:
rmse_ma_2 = np.sqrt(mean_squared_error(test_ocup, ma_2_f))
rmse_ma_3 = np.sqrt(mean_squared_error(test_ocup, ma_3_f))
rmse_ma_4 = np.sqrt(mean_squared_error(test_ocup, ma_4_f))
rmse_ma_5 = np.sqrt(mean_squared_error(test_ocup, ma_5_f))

print(rmse_ma_2, rmse_ma_3, rmse_ma_4, rmse_ma_5)

In [ ]:
# Tabla comparativa de modelos
tabla_rmse = pd.DataFrame({
    "Modelo": ["Promedio Movil"] * 4,
    "Orden (w)": [2, 3, 4, 5],
    "RMSE": [round(rmse_ma_2, 2), round(rmse_ma_3, 2), round(rmse_ma_4, 2), round(rmse_ma_5, 2)]
})
tabla_rmse["Mejor modelo"] = tabla_rmse["RMSE"] == tabla_rmse["RMSE"].min()
tabla_rmse

In [ ]:
# Grafico comparativo
fig = plt.figure(figsize=(20, 6))
plt.plot(train_ocup, label="Ocupados (entrenamiento)")
plt.plot(ma_2_f, label=f"MA(2) RMSE={rmse_ma_2:.2f}")
plt.plot(ma_3_f, label=f"MA(3) RMSE={rmse_ma_3:.2f}")
plt.plot(ma_4_f, label=f"MA(4) RMSE={rmse_ma_4:.2f}")
plt.plot(ma_5_f, label=f"MA(5) RMSE={rmse_ma_5:.2f}")
plt.plot(test_ocup, label="Real (prueba)", color="black", linewidth=2)
plt.legend()
plt.grid()
plt.title("Comparacion modelos Promedio Movil — Ocupados")
plt.show()

El mejor RMSE de Ocupados es con los ultimos **2 valores** (MA(2)).

## **5. Pronostico para los proximos 6 meses**

Se selecciona **MA(2)** por presentar el menor RMSE. Se reentrana con la serie completa.

In [ ]:
pronostico_6 = fore_ma(data[["Ocupados"]], 2, 6)

tabla_pronostico = pd.DataFrame(
    {"Ocupados_pronosticado_miles": pronostico_6["Ocupados"].values},
    index=pronostico_6.index
)
tabla_pronostico.index.name = "mes"
tabla_pronostico

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["Ocupados"], label="Serie historica")
plt.plot(pronostico_6, label="Pronostico MA(2) — 6 meses",
         color="crimson", linewidth=2, linestyle="--", marker="o")
plt.axvline(x=data.index[-1], color="gray", linestyle=":", linewidth=1.5)
plt.title("Pronostico del numero de ocupados — proximos 6 meses (MA orden 2)")
plt.xlabel("Mes")
plt.ylabel("Miles de personas")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()